<font size=10>**DATA EXPLORATION & PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. Data Preprocessing](#3) 
    - [3.1 Filtering](#31-filtering)
    - [3.2 Drop Data](#32-drop-data)
    - [3.3 Data Types](#33-data-types)
    - [3.4 Text Preprocessing](#34-text-preprocessing)
    - [3.5 Outiers](#35-outiers)
    - [3.6 Missing Values](#36-missing-values)
    - [3.7 Export Preprocessed Data](#37-export-preprocessed-data)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **Contract Timeframe**: From January 1, 2023 to April 25, 2026

In [4]:
# MERGE DATASETS
paths = {
    "2023_part01": "../data/contratos2023_part01.csv",
    "2023_part02": "../data/contratos2023_part02.csv",
    "2023_part03": "../data/contratos2023_part03.csv",
    "2024_part01": "../data/contratos2024_part01.csv",
    "2024_part02": "../data/contratos2024_part02.csv",
    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2023_part01 from ../data/contratos2023_part01.csv...
Dataset for 2023_part01 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part02 from ../data/contratos2023_part02.csv...
Dataset for 2023_part02 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part03 from ../data/contratos2023_part03.csv...
Dataset for 2023_part03 loaded successfully with shape (64562, 35).
Loading dataset for 2024_part01 from ../data/contratos2024_part01.csv...
Dataset for 2024_part01 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part02 from ../data/contratos2024_part02.csv...
Dataset for 2024_part02 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part03 from ../data/contratos2024_part03.csv...
Dataset for 2024_part03 loaded successfully with shape (75441, 35).
Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading datas

In [5]:
merged_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 732573 entries, 0 to 732572
Data columns (total 35 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   idcontrato                732573 non-null  int64  
 1   nAnuncio                  114355 non-null  str    
 2   TipoAnuncio               114355 non-null  str    
 3   idINCM                    114355 non-null  float64
 4   tipoContrato              732572 non-null  str    
 5   idprocedimento            732573 non-null  int64  
 6   tipoprocedimento          732573 non-null  str    
 7   objectoContrato           732571 non-null  str    
 8   descContrato              732573 non-null  str    
 9   adjudicante               732565 non-null  str    
 10  adjudicatarios            731904 non-null  str    
 11  dataPublicacao            732573 non-null  str    
 12  dataCelebracaoContrato    730239 non-null  str    
 13  precoContratual           732573 non-null  float64
 14 

$\rightarrow$**Columns To Keep**:

**Identifiers & Contract Info**

| column name | |
|--- | --- |
| idcontrato | |
| tipoContrato | |
| tipoFimContrato | |
| CPV | |
| tipoprocedimento | |

**Entities**

| column name | |
|--- | --- |
| adjudicante | |
| adjudicatarios | |
| concorrentes | |

**Financial Variables**

| column name | |
|--- | --- |
| precoBaseProcedimento | |
| precoContratual | |
| PrecoTotalEfetivo | |

**Location** 

| column name | |
|--- | --- |
| LocalExecucao | |

**Dates** 

| column name | |
|--- | --- |
| dataDecisaoAdjudicacao | |
| dataCelebracaoContrato | |
| dataPublicacao | |
| dataFechoContrato | |

In [6]:
cols_to_keep = [
    'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'tipoprocedimento',
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao', 
    "dataDecisaoAdjudicacao", "dataCelebracaoContrato", "dataPublicacao", "dataFechoContrato"
]

subset = merged_dataset[cols_to_keep]

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 10309 Public Entities.
There are 143783 Companies.
So, in total our analysis contains 154092 Nodes.


# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=6>**3.1 Duplicates**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [8]:
# Check duplicated rows
duplicated_rows = subset.duplicated()
print(f"Number of duplicated rows: {duplicated_rows.sum()}")

Number of duplicated rows: 2826


In [9]:
# dropping duplicated rows
subset = subset.drop_duplicates()

## <font size=6>**3.2 Missing Values**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [10]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
tipoFimContrato,579308,79.38
dataFechoContrato,576960,79.06
concorrentes,388427,53.23
dataDecisaoAdjudicacao,2334,0.32
dataCelebracaoContrato,2334,0.32
LocalExecucao,1647,0.23
adjudicatarios,658,0.09
idcontrato,0,0.00
tipoprocedimento,0,0.00
adjudicante,8,0.00


In [11]:
# rows with missing values except for 'concorrentes', 'dataFechoContrato' and 'tipoFimContrato'
subset = subset.dropna(subset=[col for col in subset.columns if col not in ['concorrentes', 'dataFechoContrato', 'tipoFimContrato']])

In [12]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 732573
Number of Contracts Now: 726745
Percentage of Deleted Contracts: 0.8%


## <font size=6>**3.3 Preprocessing Per Column**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

### <font size=6>3.3.1 Date Columns</font> <a class="anchor" id="3.3.1"></a>
  
[Back to TOC](#toc)

In [13]:
# date type conversion    
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')
subset["dataDecisaoAdjudicacao"] = pd.to_datetime(subset["dataDecisaoAdjudicacao"], errors='coerce')
subset["dataFechoContrato"] = pd.to_datetime(subset["dataFechoContrato"], errors='coerce')

In [14]:
dfs = []

date_cols = {
    'dataDecisaoAdjudicacao': 'Decision',
    'dataCelebracaoContrato': 'Celebration',
    'dataFechoContrato': 'Closure'
}

for col, label in date_cols.items():
    df = (
        subset
        .dropna(subset=[col])
        .assign(month=subset[col].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .reset_index(name='number_of_contracts')
    )
    
    df['type'] = label
    dfs.append(df)

final_df = pd.concat(dfs)

fig = px.line(
    final_df,
    x='month',
    y='number_of_contracts',
    color='type',
    title='Number of Contracts by Month (Different Dates)',
    labels={
        'month': 'Month',
        'number_of_contracts': 'Number of Contracts',
        'type': 'Date Type'
    }
)

fig.show()

In [15]:
# compute difference in months between celebration and closure, then plot histogram
months_df = subset.dropna(subset=['dataCelebracaoContrato','dataFechoContrato']).copy()
s = months_df['dataCelebracaoContrato']
e = months_df['dataFechoContrato']
months_df['months_diff'] = (e.dt.year - s.dt.year) * 12 + (e.dt.month - s.dt.month) + (e.dt.day - s.dt.day) / 30.0

fig_months = px.histogram(
    months_df,
    x='months_diff',
    nbins=100,
    title='Months between Celebration and Closure',
    labels={'months_diff': 'Months difference', 'count': 'Number of Contracts'}
)
fig_months.update_xaxes(range=[float(months_df['months_diff'].min()), float(months_df['months_diff'].max())])
fig_months.show()


### <font size=6>3.3.2 Tipo de Procedimento</font> <a class="anchor" id="3.3.2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **tipoprocedimento**: Concurso público

In [16]:
subset = subset[subset['tipoprocedimento'] == 'Concurso público']
subset.drop(columns=['tipoprocedimento'], inplace=True)

In [17]:
subset.shape

(112170, 15)

### <font size=6>3.3.3 Tipo de Contrato</font> <a class="anchor" id="3.3.3"></a>
  
[Back to TOC](#toc)

In [18]:
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [19]:
subset['tipoContrato'].value_counts()

tipoContrato
Aquisição de bens móveis                                                            60843
Aquisição de serviços                                                               31747
Empreitadas de obras públicas                                                       15860
Locação de bens móveis                                                               2062
Aquisição de bens móveis | Aquisição de serviços                                      945
Concessão de serviços públicos                                                        285
Aquisição de serviços | Locação de bens móveis                                        216
Aquisição de bens móveis | Locação de bens móveis                                      58
Aquisição de serviços | Empreitadas de obras públicas                                  37
Outros                                                                                 25
Concessão de obras públicas                                                            

In [20]:
subset_counts = subset['tipoContrato'].value_counts().reset_index()
subset_counts.columns = ['tipoContrato', 'count']

# Get top 10
top_10 = subset_counts.head(10)

fig = px.bar(
    top_10,
    x='tipoContrato',
    y='count',
    title='Number of Contracts by Contract Type',
    labels={'tipoContrato': 'Contract Type', 'count': 'Number of Contracts'}
)

fig.show()

### <font size=6>3.3.4 CPV</font> <a class="anchor" id="3.3.4"></a>
  
[Back to TOC](#toc)

In [21]:
subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [22]:
subset['CPV'].value_counts() 

CPV
Material médico de consumo                                                                                                         12977
Equipamento médico                                                                                                                  4286
Produtos farmacêuticos                                                                                                              2339
Reagentes de laboratório                                                                                                            1790
Serviços de seguros                                                                                                                 1716
                                                                                                                                   ...  
Microanalisadores de raios X                                                                                                           1
Serviços de reparação e manutenção de

In [23]:
subset_counts = subset['CPV'].value_counts().reset_index()
subset_counts.columns = ['CPV', 'count']

# Get top 10
top_10 = subset_counts.head(10)

fig = px.bar(
    top_10,
    x='CPV',
    y='count',
    title='Number of Contracts by CPV',
    labels={'CPV': 'CPV', 'count': 'Number of Contracts'}
)

fig.show()

### <font size=6>3.3.5 Concorrentes</font> <a class="anchor" id="3.3.5"></a>
  
[Back to TOC](#toc)

In [24]:
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [25]:
# number of competitors per contract
subset['nr_concorrentes'] = subset['concorrentes'].apply(
    lambda x: len([i for i in re.split(r'\s*\|\s*', x) if i]) 
    if isinstance(x, str) else 0
)

fig = px.histogram(
    subset,
    x='nr_concorrentes',
    nbins=30,
    title='Histogram of Number of Competitors'
)

fig.show()

### <font size=6>3.3.6 Adjudicante & Adjudicatário</font> <a class="anchor" id="3.3.6"></a>
  
[Back to TOC](#toc)

In [26]:
# extract contribuinte numbers from adjudicante and adjudicatarios
subset['contribuinte_adjudicante'] = subset['adjudicante'].str.extract(r'(\d{9})')
subset['adjudicante'] = subset['adjudicante'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

subset['contribuinte_adjudicatarios'] = subset['adjudicatarios'].str.extract(r'(\d{9})')
subset['adjudicatarios'] = subset['adjudicatarios'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

### <font size=6>3.3.7 Local de Execução</font> <a class="anchor" id="3.3.7"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **district**: Lisboa

In [27]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside each cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

first_location = subset['LocalExecucao'].str.split(' \| ', expand=False).str[0]

split_cols = first_location.str.split(', ', expand=True)

split_cols = split_cols.reindex(columns=[0, 1, 2])
split_cols.columns = ['country', 'district', 'city']

subset[['country', 'district', 'city']] = split_cols

for col in ['country', 'district', 'city']:
    subset[col] = subset[col].replace(r'^\s*$', pd.NA, regex=True)

# handle inconsistent structures
n_parts = first_location.str.split(', ').str.len()

# if only 1 part - it's country
subset.loc[n_parts == 1, ['district', 'city']] = pd.NA

# if 2 parts - assume country + district
subset.loc[n_parts == 2, 'city'] = pd.NA

In [28]:
subset[['country', 'district', 'city']].value_counts(dropna=False)

country          district  city   
Portugal         Lisboa    Lisboa     19885
                 NaN       NaN        16103
                 Porto     Porto       4566
                 Setúbal   Almada      2189
                 Coimbra   Coimbra     2053
                                      ...  
Israel           NaN       NaN            1
Países Baixos    NaN       NaN            1
Cabo Verde       NaN       NaN            1
Costa do Marfim  NaN       NaN            1
Ucrânia          NaN       NaN            1
Name: count, Length: 350, dtype: int64

In [29]:
subset_plot = (
    subset.dropna(subset=["district"])  # remove missing districts
      .groupby("district")
      .agg(
          n_contracts=("idcontrato", "count"),
          n_adjudicantes=("contribuinte_adjudicante", "nunique"),
          n_adjudicatarios=("contribuinte_adjudicatarios", "nunique")
      )
      .reset_index()
)

In [30]:
fig = go.Figure()

# --- traces ---
part01 = subset_plot.sort_values("n_contracts", ascending=False)
fig.add_trace(go.Bar(
    x=part01["district"],
    y=part01["n_contracts"],
    name="Contracts",
    visible=True  # default visible
))

part02 = subset_plot.sort_values("n_adjudicantes", ascending=False)
fig.add_trace(go.Bar(
    x=part02["district"],
    y=part02["n_adjudicantes"],
    name="Adjudicantes",
    visible=False
))

part03 = subset_plot.sort_values("n_adjudicatarios", ascending=False)
fig.add_trace(go.Bar(
    x=part03["district"],
    y=part03["n_adjudicatarios"],
    name="Adjudicatarios",
    visible=False
))

# --- dropdown ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Contracts",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "Number of Contracts"}]
                ),
                dict(
                    label="Adjudicantes",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "Number of Adjudicantes"}]
                ),
                dict(
                    label="Adjudicatarios",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "Number of Adjudicatarios"}]
                ),
            ],
            direction="down",
            showactive=True
        )
    ]
)

fig.update_layout(
    title="Contracts per District",
    xaxis_title="District",
    yaxis_title="Count",
    xaxis_tickangle=-45
)

fig.show()

In [31]:
subset = subset[subset['district'] == 'Lisboa']
subset.drop(columns=['LocalExecucao', 'country', 'district'], inplace=True)

In [32]:
subset['city'].value_counts()

city
Lisboa                    19885
Sintra                     2008
Cascais                    1792
Loures                     1747
Oeiras                     1572
Amadora                     689
Vila Franca de Xira         526
Mafra                       453
Torres Vedras               291
Odivelas                    237
Alenquer                    148
Cadaval                      59
Arruda dos Vinhos            58
Lourinhã                     57
Azambuja                     35
Sobral de Monte Agraço       17
Name: count, dtype: int64

### <font size=6>3.3.8 Price Columns</font> <a class="anchor" id="3.3.8"></a>
  
[Back to TOC](#toc)

In [33]:
# keep idcontrato
price_cols = ['precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo']
box_df = subset[['idcontrato'] + price_cols].copy()

# convert only price columns
box_df[price_cols] = box_df[price_cols].apply(pd.to_numeric, errors='coerce')

# melt while keeping idcontrato
long_df = box_df.melt(
    id_vars='idcontrato',
    var_name='Column',
    value_name='Value'
).dropna()

fig = px.box(
    long_df,
    x='Column',
    y='Value',
    color='Column',
    points='outliers',
    title='Distribution of Price Columns',
    hover_data=['idcontrato'] 
)

fig.update_layout(showlegend=False)
fig.show()

In [34]:
subset[subset['idcontrato'] == 10585414]

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,dataDecisaoAdjudicacao,dataCelebracaoContrato,dataPublicacao,dataFechoContrato,nr_concorrentes,contribuinte_adjudicante,contribuinte_adjudicatarios,city
107131,10585414,Empreitadas de obras públicas,NaN,Obras de construção de edifícios relacionados ...,"METROPOLITANO DE LISBOA, E.P.E","METROS. SEBASTIÃO ALCÃNTARA, ACE","--CONTRATAS Y VENTAS, S.A.U. | --COMSA Empresa...",330000000.0,321888000.0,0.0,2023-12-04,2023-12-22,2024-02-27,NaT,18,500192855,517895927,Lisboa


## <font size=6>**3.4 Final Data**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [35]:
subset.head()

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,dataDecisaoAdjudicacao,dataCelebracaoContrato,dataPublicacao,dataFechoContrato,nr_concorrentes,contribuinte_adjudicante,contribuinte_adjudicatarios,city
67,9673773,Aquisição de bens móveis,NaN,Equipamento médico,Instituto Português de Oncologia de Lisboa Fra...,"VWR Internacional - Material de Laboratório, Lda","503636975-ENZIfarma, S.A. | 503387398-Alfagene...",7517.0,234.00,0.00,2022-12-04,2023-01-02,2023-01-04,NaT,9,506361616,503842770,Lisboa
71,9674355,Empreitadas de obras públicas,"O cumprimento, a impossibilidade definitiva e ...",Obras de recuperação,Gebalis - Gestão do Arrendamento da Habitação ...,Metangular Construções Lda.,"509944647-Construbuild - Services, Limitada | ...",362000.0,57393.03,57393.03,2022-12-09,2023-01-03,2023-01-04,2023-06-29,13,503541567,510826768,Lisboa
96,9676561,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",Tubagem de revestimento e tubos,SIMAR – Serviços Intermunicipalizados de Loure...,"HUMBERTO POÇAS, S.A.",NaN,72500.0,519.00,519.00,2022-12-19,2023-01-04,2023-01-05,2023-10-04,0,680009671,501075666,Loures
155,9677495,Aquisição de bens móveis,NaN,Produtos de limpeza e polimento,"Instituto Português de Oncologia de Lisboa FG,...","João Antunes Amaro, Lda",NaN,101077.6,1272.87,0.00,2023-01-02,2023-01-02,2023-01-05,NaT,0,506361616,500149003,Lisboa
197,9670090,Empreitadas de obras públicas,"O cumprimento, a impossibilidade definitiva e ...",Obras de revisão e recuperação,Gebalis - Gestão do Arrendamento da Habitação ...,JRC – Construção e Obras Públicas SA,"510867626-PAJG - REMODELAÇÕES, UNIPESSOAL LDA ...",734549.0,689896.36,669534.39,2022-12-06,2023-01-03,2023-01-03,2023-12-29,4,503541567,502974699,Lisboa


In [36]:
subset.info()

<class 'pandas.DataFrame'>
Index: 30485 entries, 67 to 732492
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   30485 non-null  int64         
 1   tipoContrato                 30485 non-null  str           
 2   tipoFimContrato              4090 non-null   str           
 3   CPV                          30485 non-null  str           
 4   adjudicante                  30485 non-null  str           
 5   adjudicatarios               30485 non-null  str           
 6   concorrentes                 21158 non-null  str           
 7   precoBaseProcedimento        30485 non-null  float64       
 8   precoContratual              30485 non-null  float64       
 9   PrecoTotalEfetivo            30485 non-null  float64       
 10  dataDecisaoAdjudicacao       30485 non-null  datetime64[us]
 11  dataCelebracaoContrato       30485 non-null  datetime64

In [37]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 1031 Public Entities.
There are 9200 Companies.
So, in total our analysis contains 10231 Nodes.


In [38]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 732573
Number of Contracts Now: 30485
Percentage of Deleted Contracts: 95.84%


## <font color='#BFD72F' size=6>**4. Export Preprocessed Data**</font> <a class="anchor" id="4"></a>
  
[Back to TOC](#toc)

In [39]:
subset.to_csv("../data/preprocessed_data.csv", index=False)